[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# DevSecOps, entornos e IaC.

Este cuaderno funciona como capítulo introductorio de este curso. Su objetivo es fijar el lenguaje común del curso antes de entrar a pipelines, quality gates, despliegues, *Terraform* y operación segura.

## Objetivos.

- Distinguir DevOps de DevSecOps y entender por qué la distinción importa.
- Identificar las políticas de repositorio que soportan CI/CD seguro.
- Configurar validaciones locales con *pre-commit*.
- Entender el papel de los entornos `dev`, `test` y `prod` en la cadena de entrega.
- Comprender qué es IaC y cómo se relaciona con los entornos y el pipeline.

## Qué es DevSecOps.

DevSecOps es una forma de organizar el desarrollo y la operación de software en la que seguridad, calidad, entrega e infraestructura forman parte del mismo sistema de trabajo. No significa mover la seguridad a un equipo aislado al final del proceso, sino integrarla desde el diseño, el desarrollo, la validación, el build, el despliegue y la observabilidad.

En la práctica, DevSecOps implica:

- automatizar verificaciones de calidad y seguridad lo más pronto posible;
- tratar la infraestructura y las políticas como código versionado;
- reducir credenciales estáticas y privilegios excesivos;
- dejar evidencia verificable de cómo se construyó y desplegó un artefacto;
- establecer retroalimentación rápida cuando algo falla.

La idea central es que seguridad no sea una puerta al final del proceso, sino una propiedad verificable de toda la cadena de entrega.

## DevOps y DevSecOps.

DevOps pone el foco en colaboración, automatización y entrega continua entre desarrollo y operación. DevSecOps conserva esos principios, pero agrega una exigencia: toda decisión técnica relevante debe considerar impacto de seguridad y trazabilidad.

En un flujo DevSecOps maduro, preguntas como estas no se dejan para el cierre:

- quién puede promover cambios a producción;
- con qué identidad autentica el pipeline frente a la nube;
- si una imagen o paquete tiene SBOM y provenance;
- si los secretos viven en el lugar correcto;
- si la infraestructura fue creada manualmente o por código auditado.

## Esquema visual: DevSecOps como cadena de control.

```text
Cambio en repo
   |
   v
Calidad técnica -> Seguridad -> Build -> Evidencia -> Despliegue -> Observabilidad
   |                |            |         |             |               |
   |                |            |         |             |               |
 lint/tests      escaneos      imagen     SBOM      entornos        smoke tests
 tipado          secretos      o paquete  provenance  aprobaciones   rollback
```

La idea del esquema es simple: cada fase agrega una capa de confianza. Si una fase falla, la promoción debe detenerse.

## Visual: ciclo DevSecOps y cadena de herramientas.

Las siguientes imágenes complementan el esquema textual y ayudan a ubicar herramientas por fase del ciclo.

<p><strong>1) Ciclo DevSecOps.</strong></p>
<img src="img/04/devsecops.png" width="70%">

<p><strong>2) Cadena de herramientas asociada al ciclo.</strong></p>
<img src="img/04/toolchain.png" width="90%">

Referencia didáctica: el workflow que implementaremos en los siguientes cuadernos recorre estas fases con controles de calidad, seguridad, build, despliegue y observabilidad.

## Workflows de repositorio.

Un workflow de repositorio describe cómo fluye el cambio desde que alguien modifica código hasta que ese cambio puede integrarse, liberarse o desplegarse. No debe confundirse con un workflow de GitHub Actions: el primero es una política de colaboración del repositorio; el segundo es una automatización concreta dentro de esa política.

Algunos patrones comunes son:

- `feature branches`: cada cambio vive en una rama corta y se integra por pull request;
- `trunk-based development`: ramas muy cortas y merges frecuentes hacia una rama principal protegida;
- `release branches`: se separan ramas para estabilizar una versión antes de liberarla;
- `hotfix branches`: permiten corregir incidentes críticos sin mezclar todos los cambios en curso.

Para este curso se recomienda una estrategia simple: ramas cortas, pull requests, protección de rama principal y checks obligatorios antes del merge. Esa combinación favorece trazabilidad y reduce riesgo operacional.

## Políticas de repositorio que importan.

Para que un workflow de repositorio soporte CI/CD seguro, conviene definir algunas reglas explícitas:

- la rama principal no debe aceptar cambios directos sin revisión;
- los pull requests deben disparar checks mínimos de calidad;
- los workflows sensibles deben vivir bajo protección y revisión;
- las liberaciones deben quedar asociadas a tags o versiones trazables;
- los cambios de infraestructura y despliegue no deben tener el mismo nivel de libertad que un cambio de texto o documentación.

Estas reglas son operativas, no burocráticas: buscan que el repositorio refleje el nivel de control que se espera del sistema que luego será desplegado.

## Validaciones locales: *pre-commit*.

[*pre-commit*](https://pre-commit.com/) es un gestor de hooks de Git que ejecuta verificaciones automáticas en la máquina del desarrollador justo antes de que el commit quede registrado. Es la primera línea de defensa en un flujo DevSecOps: atrapar problemas antes de que salgan del equipo.

La lógica es simple: no tiene sentido enviar código con errores de formato, conflictos no resueltos o credenciales expuestas al pipeline de CI si esos errores se pueden detectar localmente en segundos.

*pre-commit* no reemplaza el CI/CD. El pipeline valida en un entorno neutral y reproducible; *pre-commit* filtra el ruido antes de que el cambio salga de la máquina. Ambas capas son necesarias.

## Instalación y configuración básica.

*pre-commit* se instala una sola vez en el entorno del desarrollador y se activa por repositorio. Se recomienda instalarlo vía [*pipx*](https://pipx.pypa.io/) para que viva en su propio entorno aislado y no contamine las dependencias del proyecto:

```bash
# instalar pre-commit de forma aislada
pipx install pre-commit

# activar los hooks en el repositorio actual (crea .git/hooks/pre-commit)
pre-commit install

# ejecutar todos los hooks manualmente sobre los archivos actuales
pre-commit run --all-files

# actualizar los hooks a las revisiones más recientes
pre-commit autoupdate
```

A partir de `pre-commit install`, cada `git commit` ejecuta automáticamente los hooks definidos en `.pre-commit-config.yaml`. Si algún hook falla, el commit se cancela y el desarrollador ve exactamente qué archivo y qué regla fallaron.

## Ejemplo de .pre-commit-config.yaml.

El archivo `.pre-commit-config.yaml` vive en la raíz del repositorio y define qué hooks se ejecutan, desde qué repositorio se obtienen y en qué versión:

```yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.6.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-json
      - id: check-merge-conflict
      - id: detect-private-key

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.4.4
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.10.0
    hooks:
      - id: mypy
```

Cada entrada en `repos` apunta a un repositorio de hooks externo. La clave `rev` fija la versión exacta, lo que garantiza reproducibilidad entre máquinas del equipo.

| Hook | Propósito | Capa |
|---|---|---|
| `trailing-whitespace` | elimina espacios al final de línea | formato |
| `end-of-file-fixer` | asegura newline al final de cada archivo | formato |
| `check-yaml` | valida sintaxis YAML | sintaxis |
| `check-json` | valida sintaxis JSON | sintaxis |
| `check-merge-conflict` | detecta marcadores de conflicto no resueltos | integridad |
| `detect-private-key` | bloquea commits con claves privadas accidentales | seguridad |
| `ruff` | linting y formato del código *Python* | calidad |
| `mypy` | validación de tipos estáticos | calidad |

## *pre-commit* en el ciclo DevSecOps.

*pre-commit* refuerza las políticas del repositorio al hacerlas ejecutables localmente. Cuando el CI/CD repite las mismas verificaciones en el pipeline, el resultado es un sistema de dos capas:

```text
desarrollador (local)              servidor (CI/CD)
       |                                  |
  pre-commit                       GitHub Actions
  - formato y estilo               - lint completo
  - sintaxis de archivos           - tests automatizados
  - secretos accidentales          - escaneos de seguridad
  - conflictos no resueltos        - build y despliegue
```

La separación es intencional: *pre-commit* es rápido y corre sin conexión; el pipeline es exhaustivo y corre en un entorno neutral. Ambas capas se complementan para que ningún cambio defectuoso llegue a producción.

## Entornos: dev, test y prod.

Los entornos separan fases del ciclo de entrega. No son solo nombres convenientes; representan niveles distintos de riesgo, estabilidad, datos, permisos y controles.

### `dev`

Es el entorno de iteración rápida. Se usa para validar cambios tempranos, explorar configuraciones y comprobar que una versión funciona en condiciones cercanas a la realidad, pero con menor costo y menor restricción.

### `test`

Es el entorno donde se verifican comportamiento, integración y criterios de aceptación con mayor disciplina. Puede incluir datos sintéticos, servicios simulados, pruebas automatizadas y validaciones más cercanas a un despliegue real.

### `prod`

Es el entorno que recibe tráfico o uso real. Por eso exige mayor estabilidad, aprobaciones, observabilidad, control de secretos, identidades acotadas y mecanismos claros de rollback.

## Esquema visual: promoción entre entornos.

```text
feature branch
      |
      v
pull request
      |
      v
checks en CI
      |
      v
merge a main
      |
      v
deploy a dev -> validación -> deploy a test -> aprobación -> deploy a prod
```

Este recorrido muestra que el mismo cambio no debería saltar directo a producción sin pasar por controles progresivos.

## Promoción entre entornos.

La promoción sana no consiste en reconstruir artefactos distintos en cada entorno, sino en promover el mismo artefacto validado a través de controles diferentes. La construcción debe ser reproducible; lo que cambia entre `dev`, `test` y `prod` son variables, permisos, aprobaciones, datos y restricciones operativas.

Ese principio evita una falla común: probar una cosa y desplegar otra.

## Tabla rápida de entornos.

| Entorno | Objetivo principal | Riesgo aceptable | Controles típicos |
| --- | --- | --- | --- |
| `dev` | iteración y validación temprana | medio | variables básicas, despliegue rápido, datos no productivos |
| `test` | validación funcional e integración | medio-bajo | pruebas automatizadas, datos sintéticos, checks adicionales |
| `prod` | operación real | bajo | aprobaciones, observabilidad, secretos controlados, rollback |

## Infraestructura como código.

Un programador *Python* ya conoce una forma de infraestructura como código aunque no la llame así: el `Dockerfile`. En lugar de documentar cómo instalar dependencias en un servidor, se escribe un archivo que cualquier máquina puede ejecutar para reproducir exactamente el mismo entorno. El resultado es predecible, versionable y revisable en un pull request.

**Infraestructura como código (IaC)** aplica la misma idea a todo lo que rodea a ese contenedor: el servidor donde corre, la red donde vive, la base de datos a la que se conecta, los permisos que tiene, las reglas de firewall que lo protegen. Todo eso, que normalmente se configura manualmente desde una consola web, se define en archivos de texto y se gestiona con las mismas herramientas que el código de la aplicación.

La motivación es la misma que llevó a adoptar control de versiones: la infraestructura configurada manualmente es opaca, difícil de auditar e imposible de reproducir con exactitud. Con el tiempo, los entornos acumulan diferencias que nadie documentó y que solo aparecen cuando algo falla en producción. IaC convierte esa configuración en algo versionable, revisable y ejecutable de forma automatizada.

### IaC y entornos.

La conexión entre IaC y los entornos `dev`, `staging` y `prod` es directa: en lugar de tres entornos configurados manualmente que acumulan diferencias invisibles con el tiempo, IaC permite definir la infraestructura una sola vez y desplegarla con parámetros distintos según el entorno. Lo que varía entre entornos no es el código de infraestructura, sino los valores que se le pasan: tamaño de instancia, nivel de logging, restricciones de red, credenciales.

Esto tiene una consecuencia importante para la promoción: si el mismo código de IaC define `dev` y `prod`, una prueba exitosa en `dev` garantiza que la infraestructura de `prod` se construirá de la misma forma. Sin IaC, esa garantía no existe.

### IaC en el pipeline.

El pipeline de CI/CD no solo valida y despliega la aplicación; también valida y aplica los cambios de infraestructura. Una modificación al servidor o a los permisos pasa por el mismo proceso de revisión, prueba y aprobación que cualquier cambio de código. El resultado es que aplicación e infraestructura evolucionan juntas, con el mismo nivel de trazabilidad.

```text
cambio de código          cambio de infraestructura
       |                          |
       v                          v
   pull request              pull request
       |                          |
       v                          v
  CI: lint + tests          CI: validate (plan)
       |                          |
       v                          v
  despliegue a dev          aplicar en dev
       |                          |
       v                          v
  promoción a prod          aplicar en prod (con aprobación)
```

[*Terraform*](https://developer.hashicorp.com/terraform) es la herramienta de IaC que este curso utiliza. Los capítulos posteriores cubren su sintaxis, gestión de estado y cómo integrarlo en el pipeline.

## Relación entre DevSecOps, repositorio, entornos e IaC.

Estos cuatro conceptos se refuerzan entre sí:

- DevSecOps define la necesidad de integrar seguridad y trazabilidad en toda la cadena;
- el workflow del repositorio define cómo se acepta y promueve el cambio;
- los entornos definen bajo qué condiciones operativas puede ejecutarse ese cambio;
- IaC garantiza que esos entornos sean reproducibles, auditables y consistentes entre sí.

Sin workflow de repositorio claro, el pipeline se vuelve inconsistente. Sin entornos claros, el despliegue pierde control. Sin IaC, los entornos divergen silenciosamente. Sin enfoque DevSecOps, la automatización puede ser rápida pero insegura.

## Cierre.

Este curso parte de esta idea: un pipeline seguro no es solo una secuencia de comandos. Es la expresión automatizada de una política técnica sobre cambios, identidades, infraestructura, artefactos y despliegues.

Los siguientes cuadernos aterrizan cada parte de ese sistema.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>